# Naive Bayes — Predição de Risco de Evasão

## 1. Objetivo

Avaliar as 3 variantes de Naive Bayes (Gaussian, Bernoulli, Complement) na predição de `alto_risco_evasao`, usando o universo de features próprio deste modelo — todas as colunas numéricas do censo, exceto vazamento (diferente dos outros 4 modelos, que usam as 21 features manualmente selecionadas; ver `docs/FEATURE_ENGINEERING.md`, seção 5, "Por que o Naive Bayes é diferente"). A variante `complement` é a vencedora da comparação original (`models/naive_bayes/artifacts/best_params.json`) e por isso é destacada, mas as 3 são treinadas e comparadas aqui — preservando "a lógica de comparação existente" pedida pela Tarefa 5.

**Escopo deste notebook:** só orquestra a chamada da pipeline
(`src.data`, `src.pipelines`, `src.training`, `src.evaluation`), carrega
os resultados estruturados e gera visualizações a partir deles.
Nenhuma lógica de carregamento de dados, engenharia de features,
definição de modelo, treino, cross-validation, busca de hiperparâmetros,
tratamento de desbalanceamento ou cálculo de métrica vive aqui — tudo
isso está em `src/` (ver `docs/ARCHITECTURE.md`, `docs/DATA_PIPELINE.md`,
`docs/FEATURE_ENGINEERING.md`, `docs/MODELS.md`, `docs/TRAINING.md`,
`docs/EVALUATION.md`).

**Reprodutibilidade:** este notebook não depende de nenhuma célula
"escondida" ou executada fora de ordem — rode `Kernel > Restart & Run
All` e ele reproduz o mesmo resultado do início ao fim, porque toda
semente aleatória vem centralizada de `src.configs.settings` através do
`Trainer`.

**Sobre o dataset:** requer `MICRODADOS_CADASTRO_CURSOS_2024.CSV`
disponível localmente (ver `README.md`, seção "Dados") — não é
versionado no Git (>100MB). Sem o arquivo, a célula da seção 3 levanta
`FileNotFoundError` com os caminhos verificados.


## 2. Configuração do experimento


In [ ]:
"""Célula de configuração de ambiente — localiza a raiz do projeto e
adiciona `src/` ao `sys.path`, para que o notebook funcione independente
de onde o Jupyter foi iniciado (raiz do repositório ou `experiments/`)."""
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "configs").is_dir():
            return candidate
    raise RuntimeError("Não foi possível localizar a raiz do projeto (pasta com src/configs).")


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Raiz do projeto: {PROJECT_ROOT}")


In [ ]:
import numpy as np
import pandas as pd

from src.data.pipeline import run_data_pipeline
from src.evaluation.comparison import fold_metrics_table
from src.evaluation.metrics import compute_metrics
from src.training.trainer import Trainer
from src.visualization.comparison_plots import plot_model_ranking
from src.visualization.evaluation_plots import (
    plot_confusion_matrix,
    plot_precision_recall_curve,
    plot_roc_curve,
)
from src.visualization.training_plots import (
    plot_metrics_boxplot,
    plot_metrics_by_fold,
    plot_threshold_by_fold,
)

pd.set_option("display.precision", 4)


In [ ]:
MODEL_NAME = "naive_bayes"
MODEL_LABEL = "Naive Bayes"
VARIANTS = ["gaussian", "bernoulli", "complement"]
WINNING_VARIANT = "complement"

# Estratégia de alvo/leakage própria do Naive Bayes (mediana de
# taxa_evasao, universo completo de colunas numéricas) — diferente dos
# outros 4 modelos. Ver docs/DATA_PIPELINE.md, seção 3.
TARGET_STRATEGY = "median_split"
MISSING_STRATEGY = "zero"
INCLUDE_PREFIX_LEAKAGE_GROUP = True

N_SPLITS = 5
RANDOM_STATE = 42


## 3. Execução do pipeline

`run_data_pipeline` executa Data Loading → Target Construction →
Leakage Removal → Base Preprocessing (`src.data`, `src.pipelines`,
Tarefas 3-4). `Trainer.train(...)` executa a engenharia de features
específica do modelo, o tratamento de desbalanceamento, o treino em
5-fold `StratifiedKFold` e o threshold tuning calibrado — tudo dentro do
fold de treino, nunca vazando para a validação (`src.training`, Tarefa
6). Nenhuma das duas chamadas abaixo tem lógica própria neste notebook.


In [ ]:
data_result = run_data_pipeline(
    target_strategy=TARGET_STRATEGY,
    missing_strategy=MISSING_STRATEGY,
    include_prefix_leakage_group=INCLUDE_PREFIX_LEAKAGE_GROUP,
)
print(f"Registros brutos: {data_result.n_records_raw:,}")
print(f"Registros após filtros do alvo: {data_result.n_records_final:,}")
print(f"Proporção de alto risco: {data_result.positive_rate:.2%}")
print(f"Features antes da engenharia específica do modelo: {data_result.X.shape[1]}")

trainer = Trainer(n_splits=N_SPLITS, random_state=RANDOM_STATE)
training_result = trainer.train(
    MODEL_NAME,
    data_result.X,
    data_result.y,
    variant=WINNING_VARIANT,

)
print("\nTreinamento concluído.")


## 4. Resultados

Métricas agregadas (média ± desvio padrão entre os 5 folds) e a tabela
por fold — ambas vêm prontas de `TrainingResult`
(`src.training.results`), sem nenhum cálculo feito neste notebook.


In [ ]:
summary_table = pd.DataFrame(
    [
        {
            "Métrica": metric_name.replace("_", " ").upper(),
            "Média": mean_value,
            "Desvio Padrão": training_result.metrics_std[metric_name],
        }
        for metric_name, mean_value in training_result.metrics.items()
    ]
)
print(f"Estratégia de desbalanceamento usada: {training_result.imbalance_strategy}")
summary_table


In [ ]:
fold_table = fold_metrics_table(training_result)
fold_table


## 5. Visualizações

Todas as figuras são geradas por `src.visualization` — este notebook só
chama as funções, passando os dados já calculados.


In [ ]:
fig_metrics = plot_metrics_by_fold(fold_table, title=f"Métricas por fold — {MODEL_LABEL}")
fig_box = plot_metrics_boxplot(fold_table, title=f"Distribuição das métricas entre folds — {MODEL_LABEL}")
fig_threshold = plot_threshold_by_fold(fold_table, title=f"Threshold calibrado por fold — {MODEL_LABEL}")


In [ ]:
# Métricas out-of-fold: para cada amostra, a predição/probabilidade do
# fold em que ela caiu como validação (nunca usada para treinar aquele
# modelo) — ver docs/TRAINING.md, seção 8.
oof_metrics = compute_metrics(training_result.y_true, training_result.predictions, training_result.probabilities)

fig_cm = plot_confusion_matrix(
    oof_metrics.confusion_matrix, title=f"Matriz de Confusão (out-of-fold) — {MODEL_LABEL}"
)
fig_roc = plot_roc_curve(training_result.y_true, training_result.probabilities, label=MODEL_LABEL)
fig_pr = plot_precision_recall_curve(training_result.y_true, training_result.probabilities, label=MODEL_LABEL)


In [ ]:
# Comparação entre as 3 variantes de Naive Bayes (preserva a lógica de
# comparação de models/naive_bayes/train.py — sem refazer a busca de
# hiperparâmetros de cada uma).
from src.evaluation.comparison import ModelComparisonRow, compare_models

variant_rows = []
for variant in VARIANTS:
    variant_result = trainer.train(MODEL_NAME, data_result.X, data_result.y, variant=variant)
    variant_rows.append(
        ModelComparisonRow(
            model_name=variant,
            mean_metrics=variant_result.metrics,
            std_metrics=variant_result.metrics_std,
        )
    )

variant_comparison = compare_models(variant_rows)
display(variant_comparison)

fig_variant_ranking = plot_model_ranking(
    variant_comparison, metric="Recall", title="Naive Bayes — Ranking das variantes por Recall"
)


## 6. Análise

A comparação entre as 3 variantes acontece na célula abaixo ("Comparação entre variantes"), treinando cada uma com `Trainer.train("naive_bayes", ..., variant=...)` — preservando a estrutura de comparação de `models/naive_bayes/train.py`, sem reimplementar a busca de ~90 configurações aleatórias (fora do escopo de `src.training`, ver `docs/TRAINING.md`, seção 6).

Diferente dos outros 4 modelos, o Naive Bayes usa SMOTE (não `class_weight`) e um universo de features maior e não manualmente selecionado — por isso não é diretamente comparável aos demais sem ressalvas (ver `ARCHITECTURE_AUDIT.md`, seção 3.5, sobre por que o Naive Bayes original não era comparável ao resto do ranking sem reavaliação).


## 7. Conclusão

Preencher após rodar com o dataset real: registrar qual variante teve melhor recall aqui e comparar com a vencedora original (`complement`, `models/naive_bayes/artifacts/best_params.json`). Comparar `training_result` (variante vencedora) com `results/audit/naive_bayes_folds.csv` — divergência é esperada, porque a auditoria já usa um pipeline de features diferente do `data/preprocessamento.py` original (ver `results/inconsistencies_report.md`, item 2).
